# Notebook 1: External Data Sources

This notebook researches available external data sources that can be used to augment the training data for Akkadian to English translation.

## 1.1 Research Overview

Based on literature review, the following external sources are available:

### Pre-trained Models
- **Helsinki-NLP/opus-mt-ROMANCE-en**: Pre-trained on Romance languages, shown to work well for Akkadian transfer learning (Nehme et al., 2025)
- **T5-base**: Used by Krueger (2023) for Akkadian/Sumerian translation
- **MarianMT**: ~75M parameter seq2seq model
- **mBART-50**: Multilingual denoising pre-training
- **NLLB**: No Language Left Behind (Meta) - supports 200 languages

### External Datasets
- **ORACC (Open Richly Annotated Cuneiform Corpus)**: Largest collection of annotated Akkadian texts
- **eBL (electronic Babylonian Library)**: Comprehensive Akkadian dictionary and texts
- **CDLI (Cuneiform Digital Library Initiative)**: Archaeological text database

## 1.2 Check Available Resources

In [ ]:
import os
import pandas as pd

# Set working directory
WORK_DIR = "/home/kwierman/Desktop/Projects/DeepPastAkkadian"
os.chdir(WORK_DIR)

# Check available data files
data_files = [
    "train.csv",
    "test.csv",
    "sample_submission.csv",
    "published_texts.csv",
    "publications.csv",
    "OA_Lexicon_eBL.csv",
    "eBL_Dictionary.csv",
    "Sentences_Oare_FirstWord_LinNum.csv"
]

print("Available data files:")
for f in data_files:
    path = os.path.join(WORK_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  ✓ {f}: {size_mb:.2f} MB")
    else:
        print(f"  ✗ {f}: NOT FOUND")

## 1.3 Analyze Supplementary Data for Augmentation

In [ ]:
# Load and analyze published_texts.csv - potential augmentation source
print("=" * 60)
print("Published Texts Analysis")
print("=" * 60)

published_texts = pd.read_csv("published_texts.csv")
print(f"Shape: {published_texts.shape}")
print(f"\nColumns: {published_texts.columns.tolist()}")
print(f"\nFirst few rows:")
published_texts.head(3)

In [ ]:
# Load and analyze publications.csv - contains OCR translations
print("=" * 60)
print("Publications Analysis (OCR output with translations)")
print("=" * 60)

publications = pd.read_csv("publications.csv")
print(f"Shape: {publications.shape}")
print(f"\nColumns: {publications.columns.tolist()}")
print(f"\nHas_Akkadian distribution:")
print(publications['has_akkadian'].value_counts())

In [ ]:
# Check lexicon data
print("=" * 60)
print("Lexicon Analysis")
print("=" * 60)

lexicon = pd.read_csv("OA_Lexicon_eBL.csv")
print(f"Shape: {lexicon.shape}")
print(f"\nColumns: {lexicon.columns.tolist()}")
print(f"\nWord types:")
print(lexicon['type'].value_counts().head(10))

## 1.4 Pre-trained Model Information

In [ ]:
# Document pre-trained model options
model_info = """
## Recommended Pre-trained Models for Akkadian → English Translation

### Primary Models

1. **Helsinki-NLP/opus-mt-ROMANCE-en**
   - Source: https://huggingface.co/Helsinki-NLP/opus-mt-ROMANCE-en
   - Architecture: MarianMT (Transformer encoder-decoder)
   - Parameters: ~75M
   - Pre-trained on: French, Spanish, Portuguese, Italian, Romanian → English
   - Best for: Transfer learning to Akkadian (Semitic language close to Romance morphologically)
   - Prior result: BLEU ~37 on Akkadian T2E task (Nehme et al., 2025)

2. **t5-base**
   - Source: https://huggingface.co/t5-base
   - Architecture: T5 (Encoder-Decoder)
   - Parameters: ~220M
   - Pre-trained on: C4 corpus (multitask)
   - Used by: Krueger (2023) for Akkadian/Sumerian

3. **facebook/mbart-large-50-many-to-many**
   - Source: https://huggingface.co/facebook/mbart-large-50-many-to-many
   - Architecture: mBART
   - Parameters: ~610M
   - Pre-trained on: 50 languages

### Installation
```bash
pip install transformers sentencepiece sacrebleu
```
"""

print(model_info)

## 1.5 Summary and Recommendations

In [ ]:
summary = """
## Summary: External Data Sources

### Available In-Project Data
- Training data: ~1,500 document-level translations
- Published texts: ~8,000 texts without translations
- Publications: ~900 PDFs with OCR text (potential for extraction)
- Lexicon: ~39,000 Old Assyrian words with lemmatization

### Recommended Approach
1. **Primary model**: Fine-tune Helsinki-NLP/opus-mt-ROMANCE-en on provided training data
2. **Secondary model**: T5-base for ensemble diversity
3. **Data augmentation**: Extract additional sentence pairs from publications.csv via text matching

### Key Considerations
- Training data is document-level, need sentence-level alignment for test
- Special handling required for: <gap>, <big_gap>, determinatives {d}, {ki}, etc.
- Normalization needed: Ḫ→H (training has Ḫ, test has H)
"""

print(summary)

In [ ]:
# Save model recommendations to file for reference
with open("notebooks/model_recommendations.md", "w") as f:
    f.write(model_info)
    f.write("\n\n")
    f.write(summary)

print("Model recommendations saved to notebooks/model_recommendations.md")